# openenv-dsc-co: grpo training on kaggle

end-to-end grpo run on kaggle t4x2 or p100. loads llama-3.2-3b-instruct (4-bit), fine-tunes with trl grpo against the in-process dsc environment, and evaluates before / after on tier 1 and tier 2 scenarios.

**kaggle settings**: accelerator = `t4 x2` or `p100`; internet = on; persistence = variables & files.

if you hit vram pressure, lower `NUM_GEN` or `MAX_SEQ`.

### what changed (april 2026)
- **no kernel restart required** — uses `unsloth[kaggle-new]` which auto-manages torch/transformers/trl compatibility
- **latest stack** — unsloth 2026.x, trl 1.2.x, transformers 5.x
- **single install cell** — replaces the old 8+ `pip install --force-reinstall` commands
- **T4/P100 safe** — uses `fast_inference=False` to avoid vLLM torch.compile crash on SM < 8.0
- **llama-3.2-3b-instruct** — instruction-tuned 3B model, fits T4 comfortably with room for GRPO rollouts

## 1. fetch the env code
two paths: hf space or github. pick one via env var.

In [ ]:
import os, subprocess, sys

REPO_DIR = '/kaggle/working/dsc'

if not os.path.isdir(REPO_DIR):
    HF_SPACE = os.environ.get('DSC_HF_SPACE', '')
    if HF_SPACE:
        subprocess.check_call(['git', 'clone', f'https://huggingface.co/spaces/{HF_SPACE}', REPO_DIR])
    else:
        subprocess.check_call(['git', 'clone', 'https://github.com/CYCLOP5/metascaler-hack', REPO_DIR])
else:
    print(f'{REPO_DIR} already exists, skipping clone')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('cwd', os.getcwd())
print('files', os.listdir('.'))

## 2. install training deps

uses `unsloth[kaggle-new]` which auto-installs compatible torch, transformers, trl, bitsandbytes. **no kernel restart needed.** no vllm needed since T4 uses `fast_inference=False`.

In [ ]:
%%capture install_output
import subprocess, sys

# --- core: unsloth with kaggle-optimised dep resolution ---
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git'],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('kaggle-new extra failed, falling back to bare unsloth install...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'unsloth'])

# --- project deps (not covered by unsloth extras) ---
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--upgrade',
    'pulp', 'pydantic', 'fastapi', 'uvicorn[standard]', 'fastmcp',
    'gymnasium', 'openenv-core', 'trackio',
])

# --- system dep for milp solver ---
subprocess.run(['apt-get', 'install', '-y', 'coinor-cbc'],
               capture_output=True)

print('install done')

In [ ]:
# show last 40 lines of install output for debugging if needed
for line in install_output.stdout.strip().split('\n')[-40:]:
    print(line)

## 3. verify environment

In [ ]:
import importlib, importlib.metadata as _im, subprocess, sys

_test = subprocess.run(
    [sys.executable, '-c',
     'import numpy; import torch; import transformers; import trl; '
     'print(f"numpy={numpy.__version__} torch={torch.__version__} '
     'transformers={transformers.__version__} trl={trl.__version__}")'],
    capture_output=True, text=True, timeout=120
)
if _test.returncode != 0:
    print('\u26a0\ufe0f  subprocess import test FAILED — you may need one manual kernel restart')
    print(_test.stderr[-800:])
else:
    print('\u2713 subprocess import test passed')
    print(' ', _test.stdout.strip())

import numpy as np
import torch
try:
    import pulp, pydantic
except ImportError as e:
    print(f'missing dep: {e}')

print()
print(f'GPU           {torch.cuda.get_device_name()}  SM {".".join(map(str, torch.cuda.get_device_capability()))}')
print(f'torch         {torch.__version__}  cuda={torch.cuda.is_available()}  gpus={torch.cuda.device_count()}')
print(f'numpy         {np.__version__}')
print(f'transformers  {_im.version("transformers")}')
print(f'trl           {_im.version("trl")}')
print(f'pydantic      {_im.version("pydantic")}')
print(f'pulp          {_im.version("PuLP")}')
try:
    print(f'unsloth       {_im.version("unsloth")}')
except _im.PackageNotFoundError:
    print('ERROR: unsloth not installed')

## 4. sanity-check env before training

In [ ]:
from server.policies import zero_op_rollout, greedy_rollout, optimal_replay_rollout
import json

baselines = {}
for tier in [1, 2]:
    for name, fn in [('zero_op', zero_op_rollout), ('greedy', greedy_rollout), ('opt_replay', optimal_replay_rollout)]:
        r = fn(seed=7, difficulty=tier)
        baselines[f'tier{tier}/{name}'] = {'gap': r.gap, 'terminal': r.terminal, 'agent': r.agent_cost, 'opt': r.optimal_cost}
print(json.dumps(baselines, indent=2))

## 5. configure training run

In [ ]:
import os

MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit'
MAX_SEQ = 4096
LORA_R = 32
DIFFICULTY = 1
NUM_GEN = 8
DATASET_SIZE = 500
MAX_STEPS = 120

os.environ['DSC_MODEL'] = MODEL_NAME
os.environ['DSC_MAX_SEQ'] = str(MAX_SEQ)
os.environ['DSC_LORA_R'] = str(LORA_R)
os.environ['DSC_TIER'] = str(DIFFICULTY)
os.environ['DSC_NUM_GEN'] = str(NUM_GEN)
os.environ['DSC_DATA_N'] = str(DATASET_SIZE)

print('config:', MODEL_NAME, 'tier=', DIFFICULTY, 'num_gen=', NUM_GEN, 'data_n=', DATASET_SIZE)

## 6. load model (unsloth 4-bit)

**`fast_inference=False`** — vLLM v0.19 torch.compile is broken on T4/P100 (SM < 8.0). this uses HF generate instead, which is slower but stable. on ampere+ GPUs (A100, L4) you can switch to `fast_inference=True`.

In [ ]:
import torch
from unsloth import FastLanguageModel, PatchFastRL

PatchFastRL("GRPO", FastLanguageModel)

# T4/P100 (SM<8): fast_inference=False to skip vLLM torch.compile crash
# A100/L4/H100 (SM>=8): switch to fast_inference=True for 3-5x faster rollouts
_use_fast = torch.cuda.get_device_capability()[0] >= 8
print(f'fast_inference={_use_fast} (SM {".".join(map(str, torch.cuda.get_device_capability()))})')

_load_kwargs = dict(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    fast_inference=_use_fast,
)
if _use_fast:
    _load_kwargs['gpu_memory_utilization'] = 0.55
    _load_kwargs['max_lora_rank'] = LORA_R

model, tokenizer = FastLanguageModel.from_pretrained(**_load_kwargs)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_R,
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
print('model ready')

## 7. grpo training loop

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from train import DSCToolEnv, reward_func, schema_reward_func, SYSTEM_PROMPT

prompts = [[
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'plan the supply chain. call tools to minimize cost.'},
]] * DATASET_SIZE
dataset = Dataset.from_dict({'prompt': prompts})

config = GRPOConfig(
    output_dir='/kaggle/working/grpo_dsc_co',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    num_generations=NUM_GEN,
    max_completion_length=2048,
    beta=0.04,
    temperature=0.7,
    use_vllm=_use_fast,
    vllm_mode='colocate' if _use_fast else None,
    log_completions=True,
    logging_steps=1,
    save_steps=40,
    max_steps=MAX_STEPS,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=config,
    train_dataset=dataset,
    reward_funcs=[reward_func, schema_reward_func],
    environment_factory=DSCToolEnv,
)
trainer.train()

## 8. save lora + upload to hf hub

In [ ]:
OUT = '/kaggle/working/grpo_dsc_co/final'
trainer.save_model(OUT)
print('saved lora', OUT)

HF_REPO = os.environ.get('DSC_HF_REPO', '')
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_REPO and HF_TOKEN:
    from huggingface_hub import HfApi
    HfApi().create_repo(HF_REPO, token=HF_TOKEN, repo_type='model', exist_ok=True)
    HfApi().upload_folder(folder_path=OUT, repo_id=HF_REPO, repo_type='model', token=HF_TOKEN)
    print('pushed', HF_REPO)

## 9. eval trained policy vs baselines

In [ ]:
import json, statistics
from train import DSCToolEnv, SYSTEM_PROMPT
from server.dsc_environment import HORIZON

FastLanguageModel.for_inference(model)

def rollout_policy(seed, difficulty):
    envw = DSCToolEnv()
    os.environ['DSC_TIER'] = str(difficulty)
    envw.reset()
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'plan the supply chain. call tools to minimize cost.'},
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', tokenize=True, add_generation_prompt=True).to(model.device)
    out = model.generate(inputs, max_new_tokens=2048, temperature=0.3, do_sample=True)
    return {'seed': seed, 'difficulty': difficulty, 'cumulative': envw.cumulative, 'terminal': envw.terminal}

rewards = []
for s in range(5):
    rewards.append(rollout_policy(s, 1))
print('trained policy tier1:', json.dumps(rewards, indent=2))